# 84: Entry Optimization (With Proven Exit)

**🎯 PROVEN EXIT from Notebook 83:**
- **MVRV > 2.0 AND Price < 50-day MA**
- 2784% return (3.3x better than Never Exit)
- 93.8% win rate over 16 trades
- Works in both 2020-2022 and 2023-2026

## The Challenge:

Current entry (Buy The Dip 4/5) had **0 trades in 2013-2018**. Too conservative?

**Performance when it DOES enter:**
- 2020-2022: +346% vs +130% B&H (2.7x better)
- 2023-2026: +746% vs +441% B&H (1.7x better)

## Goal: More Entries, Same Quality

Test different entry strategies to:
1. Get earlier entries (capture more opportunities)
2. Maintain high win rate (don't sacrifice quality)
3. Keep the proven exit working

## Important: Data Requirements

**Backtest starts when ALL data is available:**
- Some strategies use funding rates & liquidations (Glassnode)
- These typically start around 2018-2019
- Starting from when all data exists ensures fair comparison
- No strategy gets handicapped by missing data

## Entry Strategies to Test:

### A. Simpler Valuation Entries
1. **STH-MVRV < 1.0** (single indicator)
2. **MVRV < 1.0** (broader market valuation)
3. **STH-MVRV < 1.2** (slightly relaxed)
4. **Relaxed Buy The Dip** (3/5 conditions instead of 4/5)

### B. Momentum-Based Entries
5. **Price crosses above 50MA** (trend reversal)
6. **Price < 200MA** (buying in downtrend)
7. **Golden Cross** (50MA crosses above 200MA)
8. **Price < 100MA AND STH-MVRV < 1.0** (hybrid)

### C. Combined Strategies
9. **STH-MVRV < 1.0 OR Price < 200MA** (valuation OR momentum)
10. **STH-MVRV < 1.0 AND Price > 50MA** (buying dips in uptrend)
11. **SOPR < 1.0** (capitulation only)
12. **Always Invested** (baseline comparison)

**All strategies use the same exit: MVRV>2.0 AND Price<50MA**

This ensures we're comparing entry quality, not exit quality.

In [ ]:
# Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

try:
    import vectorbt as vbt
    print("✓ VectorBT loaded")
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "vectorbt"])
    import vectorbt as vbt

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

print("✓ Setup complete")

In [ ]:
# Configuration
PROJECT_ROOT = Path().resolve().parent
DATA_DIR = PROJECT_ROOT / "data" / "brk" / "daily"
GLASSNODE_DIR = PROJECT_ROOT / "data" / "glassnode" / "daily"

FEES = 0.001
SLIPPAGE = 0.001

## 1. Load Data

In [ ]:
def load_metric(name: str, source: str = "brk") -> pd.Series:
    path = DATA_DIR / f"{name}.parquet" if source == "brk" else GLASSNODE_DIR / f"{name}.parquet"
    if not path.exists():
        return pd.Series(dtype=float)
    df = pd.read_parquet(path)
    if 'time' not in df.columns and isinstance(df.index, pd.DatetimeIndex):
        df = df.reset_index()
        if len(df.columns) == 2:
            df.columns = ['time', 'value']
    if 'value' not in df.columns:
        for col in df.columns:
            if col != 'time' and pd.api.types.is_numeric_dtype(df[col]):
                df['value'] = df[col]
                break
    if 'time' in df.columns and 'value' in df.columns:
        df['time'] = pd.to_datetime(df['time'])
        return df.set_index('time')['value'].sort_index()
    return pd.Series(dtype=float)

print("Loading data...")

metrics = {
    'price': ('price', 'brk'),
    'mvrv': ('mvrv', 'brk'),
    'mvrv_sth': ('mvrv_sth', 'brk'),
    'sopr_sth': ('sopr_sth', 'brk'),
    'sopr_lth': ('sopr_lth', 'brk'),
    'realized_profit': ('realized_profit', 'brk'),
    'realized_loss': ('realized_loss', 'brk'),
    'funding': ('funding_rate', 'glassnode'),
    'liq_long': ('liquidations_long', 'glassnode'),
    'liq_short': ('liquidations_short', 'glassnode'),
}

df_dict = {name: load_metric(metric, source) for name, (metric, source) in metrics.items()}
df = pd.DataFrame(df_dict).fillna(method='ffill')

# Find when derivatives data (funding, liquidations) becomes available
funding_start = df['funding'].first_valid_index()
liq_long_start = df['liq_long'].first_valid_index()
liq_short_start = df['liq_short'].first_valid_index()

# Use the latest start date to ensure all data is available
derivatives_starts = [d for d in [funding_start, liq_long_start, liq_short_start] if d is not None]
if derivatives_starts:
    BACKTEST_START = max(derivatives_starts)
    print(f"\n📊 Derivatives data availability:")
    print(f"   Funding rate: {funding_start.date() if funding_start else 'N/A'}")
    print(f"   Long liquidations: {liq_long_start.date() if liq_long_start else 'N/A'}")
    print(f"   Short liquidations: {liq_short_start.date() if liq_short_start else 'N/A'}")
    print(f"   → Starting backtest from: {BACKTEST_START.date()}")
else:
    # Fallback to 2013 if no derivatives data
    BACKTEST_START = '2013-01-01'
    print(f"\n⚠️  No derivatives data found, starting from {BACKTEST_START}")

# Filter dataframe
df = df[df.index >= BACKTEST_START].copy()

print(f"\n✓ Data: {len(df)} days ({df.index[0].date()} to {df.index[-1].date()})")
print(f"  All entry strategies have complete data for fair comparison")
df.head()

## 2. Calculate Indicators

In [ ]:
print("Calculating indicators...\n")

price = df['price']

# Moving averages
df['ma_50'] = price.rolling(50).mean()
df['ma_100'] = price.rolling(100).mean()
df['ma_200'] = price.rolling(200).mean()

# Price position
df['above_ma_50'] = price > df['ma_50']
df['above_ma_100'] = price > df['ma_100']
df['above_ma_200'] = price > df['ma_200']

# MA crossovers
df['golden_cross'] = df['ma_50'] > df['ma_200']

# Price crosses
df['price_cross_above_50ma'] = (price > df['ma_50']) & (price.shift(1) <= df['ma_50'].shift(1))
df['price_cross_above_200ma'] = (price > df['ma_200']) & (price.shift(1) <= df['ma_200'].shift(1))

print("✓ Indicators calculated")

## 3. Define Entry Strategies

In [ ]:
print("Generating entry strategies...\n")

# Original Buy The Dip conditions
c1 = df['mvrv_sth'] < 1.0
c2 = df['sopr_sth'] < 1.0
c3 = (df['realized_profit'] / df['realized_loss']) < 1.0
c4 = (df['funding'] <= 0.0).fillna(False)
c5 = ((df['liq_long'] / df['liq_short']) > 1.0).fillna(False)

entry_strategies = {}

# A. Simpler Valuation Entries
entry_strategies['sth_mvrv'] = (c1.astype(bool), 'STH-MVRV < 1.0')
entry_strategies['mvrv'] = ((df['mvrv'] < 1.0).fillna(False).astype(bool), 'MVRV < 1.0')
entry_strategies['sth_mvrv_relax'] = ((df['mvrv_sth'] < 1.2).fillna(False).astype(bool), 'STH-MVRV < 1.2')
entry_strategies['btd_3of5'] = (((c1.astype(int) + c2.astype(int) + c3.astype(int) + c4.astype(int) + c5.astype(int)) >= 3).astype(bool), 'Buy The Dip (3/5)')
entry_strategies['btd_4of5'] = (((c1.astype(int) + c2.astype(int) + c3.astype(int) + c4.astype(int) + c5.astype(int)) >= 4).astype(bool), 'Buy The Dip (4/5) - Original')

# B. Momentum-Based Entries
entry_strategies['cross_50ma'] = (df['price_cross_above_50ma'].fillna(False).astype(bool), 'Price crosses above 50MA')
entry_strategies['below_200ma'] = ((~df['above_ma_200']).fillna(False).astype(bool), 'Price < 200MA')
entry_strategies['golden_cross'] = (df['golden_cross'].fillna(False).astype(bool), 'Golden Cross (50MA>200MA)')

# C. Combined Strategies
entry_strategies['sth_or_ma200'] = ((c1 | ~df['above_ma_200']).fillna(False).astype(bool), 'STH-MVRV<1.0 OR Price<200MA')
entry_strategies['sth_and_uptrend'] = ((c1 & df['above_ma_50']).fillna(False).astype(bool), 'STH-MVRV<1.0 AND Price>50MA')
entry_strategies['sopr_only'] = (c2.astype(bool), 'SOPR-STH < 1.0 (capitulation)')
entry_strategies['sth_below100ma'] = ((c1 & ~df['above_ma_100']).fillna(False).astype(bool), 'STH-MVRV<1.0 AND Price<100MA')

# D. Always invested (for comparison)
entry_strategies['always'] = (pd.Series(True, index=df.index, dtype=bool), 'Always Invested (Buy & Hold)')

print("Entry Strategies:")
print("="*80)
for key, (entries, name) in entry_strategies.items():
    print(f"{name:<50} {entries.sum():>6} signals")
print("="*80)

## 4. Define Exit Strategy (Proven Winner)

In [ ]:
# Proven exit from notebook 83
mvrv_condition = (df['mvrv'] > 2.0).fillna(False)
ma_condition = (~df['above_ma_50']).fillna(False)
proven_exit = (mvrv_condition & ma_condition).astype(bool)

print(f"Proven Exit: MVRV>2.0 AND Price<50MA")
print(f"Exit signals: {proven_exit.sum()}")

## 5. Backtest All Entry Strategies

In [ ]:
def backtest_strategy(df, entries, exits, name):
    """Backtest with proven exit."""
    try:
        pf = vbt.Portfolio.from_signals(
            close=df['price'],
            entries=entries,
            exits=exits,
            fees=FEES,
            slippage=SLIPPAGE,
            init_cash=10000,
            freq='1D'
        )
        return {
            'name': name,
            'portfolio': pf,
            'total_return': pf.total_return() * 100,
            'sharpe': pf.sharpe_ratio(),
            'max_dd': pf.max_drawdown() * 100,
            'num_trades': pf.trades.count(),
            'win_rate': pf.trades.win_rate() * 100 if pf.trades.count() > 0 else 0,
        }
    except Exception as e:
        print(f"Error backtesting {name}: {e}")
        return None

print("\n" + "="*100)
print("BACKTESTING: ENTRY OPTIMIZATION (WITH PROVEN EXIT)")
print("="*100)

results = {}
for key, (entries, name) in entry_strategies.items():
    result = backtest_strategy(df, entries, proven_exit, name)
    if result:
        results[key] = result

# Buy and hold
bh_return = (df['price'].iloc[-1] / df['price'].iloc[0] - 1) * 100

# Sort by return
sorted_results = sorted(results.items(), key=lambda x: x[1]['total_return'], reverse=True)

# Results table
print(f"\n{'Strategy':<50} {'Return':>12} {'Sharpe':>8} {'MaxDD':>10} {'Trades':>8} {'WinRate':>10}")
print("-"*100)

for key, res in sorted_results:
    print(f"{res['name']:<50} {res['total_return']:>11.1f}% {res['sharpe']:>8.2f} {res['max_dd']:>9.1f}% {int(res['num_trades']):>8} {res['win_rate']:>9.1f}%")

print(f"{'Pure Buy & Hold (no exits)':<50} {bh_return:>11.1f}% {'~1.0':>8} {'?':>10} {'-':>8} {'-':>10}")
print("="*100)

# Highlight best
best = max(results.values(), key=lambda x: x['total_return'])
best_sharpe = max(results.values(), key=lambda x: x['sharpe'])

print(f"\n🏆 BEST RETURN: {best['name']} at {best['total_return']:.1f}%")
print(f"📊 BEST SHARPE: {best_sharpe['name']} at {best_sharpe['sharpe']:.2f}")

## 6. Performance by Market Cycle

In [ ]:
# Test top 5 strategies by period
periods = [
    ('2013-01-01', '2015-12-31', '2013-2015 (Early)'),
    ('2017-01-01', '2018-12-31', '2017-2018 Cycle'),
    ('2020-01-01', '2022-12-31', '2020-2022 Cycle'),
    ('2023-01-01', '2026-01-22', '2023-2026 Bull'),
]

print("\n" + "="*100)
print("PERFORMANCE BY MARKET CYCLE (Top 5 Strategies)")
print("="*100)

# Get top 5
top_5_keys = [key for key, _ in sorted_results[:5]]

for start, end, label in periods:
    period_df = df[(df.index >= start) & (df.index <= end)].copy()
    if len(period_df) < 30:
        continue
    
    period_exit = proven_exit[(proven_exit.index >= start) & (proven_exit.index <= end)]
    
    # Buy and hold
    bh = (period_df['price'].iloc[-1] / period_df['price'].iloc[0] - 1) * 100
    
    print(f"\n{label}:")
    print(f"  Buy & Hold: {bh:+.1f}%")
    
    period_results = {}
    for key in top_5_keys:
        period_entries = entry_strategies[key][0][(entry_strategies[key][0].index >= start) & (entry_strategies[key][0].index <= end)]
        
        try:
            pf = vbt.Portfolio.from_signals(
                close=period_df['price'], entries=period_entries, exits=period_exit,
                fees=FEES, slippage=SLIPPAGE, init_cash=10000, freq='1D'
            )
            ret = pf.total_return() * 100
            trades = pf.trades.count()
            period_results[key] = (ret, trades)
        except:
            period_results[key] = (0, 0)
    
    # Sort and display
    for key, (ret, trades) in sorted(period_results.items(), key=lambda x: x[1][0], reverse=True):
        name = entry_strategies[key][1]
        print(f"  {name}: {ret:+.1f}% ({ret - bh:+.1f}% vs B&H) | {trades} trades")

print("\n" + "="*100)

## 7. Equity Curves (Top 5)

In [ ]:
# Plot top 5
fig, ax = plt.subplots(figsize=(16, 8))

# Buy & Hold
bh_equity = (df['price'] / df['price'].iloc[0]) * 10000
ax.plot(bh_equity.index, bh_equity.values, label='Buy & Hold', linewidth=3, color='gray', linestyle='--', alpha=0.7)

# Top 5 strategies
colors = ['green', 'blue', 'orange', 'red', 'purple']
for (key, res), color in zip(sorted_results[:5], colors):
    equity = res['portfolio'].value()
    ax.plot(equity.index, equity.values, label=res['name'], linewidth=2, alpha=0.8, color=color)

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Portfolio Value ($)', fontsize=12)
ax.set_title('Entry Optimization: Top 5 Strategies', fontsize=14, fontweight='bold')
ax.set_yscale('log')
ax.legend(fontsize=10, loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Final Verdict

In [ ]:
print("\n" + "="*90)
print("FINAL VERDICT: ENTRY OPTIMIZATION")
print("="*90)

best = max(results.values(), key=lambda x: x['total_return'])
original_btd = results.get('btd_4of5')

print(f"\n1. BEST ENTRY STRATEGY:")
print(f"   {best['name']}")
print(f"   Return: {best['total_return']:.1f}%")
print(f"   Sharpe: {best['sharpe']:.2f}")
print(f"   Max DD: {best['max_dd']:.1f}%")
print(f"   Trades: {int(best['num_trades'])}")
print(f"   Win Rate: {best['win_rate']:.1f}%")

if original_btd:
    print(f"\n2. COMPARISON TO ORIGINAL (Buy The Dip 4/5):")
    print(f"   Original: {original_btd['total_return']:.1f}%")
    print(f"   Best: {best['total_return']:.1f}%")
    print(f"   Improvement: {best['total_return'] - original_btd['total_return']:+.1f}%")
    print(f"   Trades: {int(original_btd['num_trades'])} → {int(best['num_trades'])} ({int(best['num_trades']) - int(original_btd['num_trades']):+d})")

print(f"\n3. TOP 5 STRATEGIES:")
for i, (key, res) in enumerate(sorted_results[:5], 1):
    print(f"   {i}. {res['name']}: {res['total_return']:.1f}% | {int(res['num_trades'])} trades | {res['win_rate']:.1f}% WR")

print(f"\n4. KEY INSIGHTS:")
if best['num_trades'] > original_btd['num_trades']:
    print(f"   ✅ Found entries that trade more frequently")
    print(f"   💡 More opportunities to capture with proven exit")
else:
    print(f"   Original entry strategy was already good")

if best['total_return'] > original_btd['total_return'] * 1.1:
    print(f"   ✅ Significant improvement (>10%) in returns")
elif best['total_return'] > original_btd['total_return']:
    print(f"   ✓ Modest improvement in returns")
else:
    print(f"   Original was already optimal")

print(f"\n5. RECOMMENDATION:")
if best['total_return'] > original_btd['total_return'] * 1.2 and best['win_rate'] > 85:
    print(f"   🎯 USE THIS ENTRY: {best['name']}")
    print(f"   ✅ Major improvement in performance")
    print(f"   ✅ Maintains high win rate")
elif best['total_return'] > original_btd['total_return'] * 1.1:
    print(f"   ✓ CONSIDER: {best['name']}")
    print(f"   Good improvement, validate in paper trading")
else:
    print(f"   ✓ STICK WITH ORIGINAL: Buy The Dip (4/5)")
    print(f"   Already well-optimized")

print(f"\n💡 PRODUCTION STRATEGY:")
print(f"   Entry: {best['name']}")
print(f"   Exit: MVRV>2.0 AND Price<50MA")
print(f"   Expected: {best['total_return']:.1f}% return, {best['win_rate']:.1f}% win rate")

print("\n" + "="*90)

## Summary

This notebook tests 13 different entry strategies with the proven exit (MVRV>2.0 + Price<50MA).

**Goal:** Find simpler or more frequent entries while maintaining quality.

**Key Questions:**
1. Can we get more trades with simpler entries?
2. Do momentum-based entries (MA crosses) work better?
3. Is relaxing Buy The Dip from 4/5 to 3/5 beneficial?
4. What's the optimal entry to pair with our proven exit?

The answer determines your production trading strategy.